# Firecrawl MCP Server Lab

Firecrawl is a powerful web scraping and search tool that can be used with MCP.

- Official docs: https://docs.firecrawl.dev/mcp-server
- GitHub: https://github.com/firecrawl/firecrawl-mcp-server

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display

load_dotenv(override=True)

True

## Step 1: Check API Key

In [2]:
firecrawl_api_key = os.getenv("FIRECRAWL_API_KEY")
if firecrawl_api_key:
    print(f"Firecrawl API Key found: {firecrawl_api_key[:10]}...")
else:
    print("FIRECRAWL_API_KEY is not set in .env file")

Firecrawl API Key found: fc-ebd9863...


## Step 2: Setup Firecrawl MCP Server and List Tools

In [3]:
params = {
    "command": "npx",
    "args": ["-y", "firecrawl-mcp"],
    "env": {"FIRECRAWL_API_KEY": firecrawl_api_key}
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

print(f"Found {len(mcp_tools)} tools:")
for tool in mcp_tools:
    print(f"  - {tool.name}: {tool.description[:80]}..." if len(tool.description) > 80 else f"  - {tool.name}: {tool.description}")

Found 8 tools:
  - firecrawl_scrape: 
Scrape content from a single URL with advanced options. 
This is the most power...
  - firecrawl_map: 
Map a website to discover all indexed URLs on the site.

**Best for:** Discover...
  - firecrawl_search: 
Search the web and optionally extract content from search results. This is the ...
  - firecrawl_crawl: 
 Starts a crawl job on a website and extracts content from all pages.
 
 **Best...
  - firecrawl_check_crawl_status: 
Check the status of a crawl job.

**Usage Example:**
```json
{
  "name": "firec...
  - firecrawl_extract: 
Extract structured information from web pages using LLM capabilities. Supports ...
  - firecrawl_agent: 
Autonomous web data gathering agent. Describe what data you want, and the agent...
  - firecrawl_agent_status: 
Check the status of an agent job.

**Usage Example:**
```json
{
  "name": "fire...


## Step 3: Test - Scrape a Web Page

In [ ]:
instructions = "You are a helpful assistant that can scrape and analyze web pages. Provide concise summaries."
request = "Please scrape https://news.ycombinator.com and tell me what the top 3 stories are about."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="scraper", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("firecrawl-scrape"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Step 4: Test - Web Search

In [ ]:
instructions = "You are a research assistant that can search the web and summarize findings."
request = "Search for the latest news about OpenAI and summarize the key points."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="researcher", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("firecrawl-search"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Step 5: Search with Ranked Results

Use Firecrawl's search tool to get ranked results on a topic.

In [4]:
# Search for a topic and return ranked results
instructions = """You are a research assistant with web search capabilities.
When searching, use the firecrawl_search tool and present results in a ranked format with:
1. Title
2. URL
3. Brief description
List the top 5 most relevant results."""

request = "Search for 'best Python web frameworks 2025' and show me the top ranked results."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="searcher", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("firecrawl-ranked-search"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Here are the top 5 ranked results for "best Python web frameworks 2025":

1. Title: The Most Popular Python Frameworks and Libraries in 2025
   URL: https://blog.jetbrains.com/pycharm/2025/09/the-most-popular-python-frameworks-and-libraries-in-2025/
   Description: Lists popular Python frameworks in 2025, including FastAPI, Django, Flask, Requests, Asyncio, and Django REST Framework.

2. Title: 2025's Top 10 Python Web Frameworks Compared - DEV Community
   URL: https://dev.to/leapcell/top-10-python-web-frameworks-compared-3o82
   Description: Compares top Python web frameworks in 2025 such as Django, FastAPI, Flask, Django REST framework, Tornado, and Sanic.

3. Title: Which Is the Best Python Web Framework: Django, Flask, or FastAPI?
   URL: https://blog.jetbrains.com/pycharm/2025/02/django-flask-fastapi/
   Description: Discusses the three consistently popular Python web frameworks: Django, Flask, and FastAPI based on Python Developer Survey insights.

4. Title: Which web framework is “king”? : r/Python - Reddit
   URL: https://www.reddit.com/r/Python/comments/12jnsh3/which_web_framework_is_king/
   Description: Community discussion mentioning Blacksheep, Sanic, and other frameworks, focusing on speed, maturity, and async features.

5. Title: Let's Settle This: Best Python Web Framework. FIGHT. #162829
   URL: https://github.com/orgs/community/discussions/162829
   Description: A GitHub community discussion suggesting no single "best" framework exists, but FastAPI is a top choice for 2025 while Django remains strong in enterprise.

If you want, I can provide more detailed summaries or specific insights from any of these sources.

## Step 6: Search + Scrape Combo

Search for results, then scrape the top result for detailed content.

In [ ]:
# Search then scrape the most relevant result
instructions = """You are a research assistant. 
1. First, use firecrawl_search to find relevant results
2. Then, use firecrawl_scrape to get detailed content from the most relevant result
3. Summarize the key information from that page"""

request = "Find and summarize the official FastAPI documentation on dependency injection."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=90) as mcp_server:
    agent = Agent(name="deep_researcher", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("firecrawl-search-scrape"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Step 7: Custom Exercise

Try your own search or scraping task!

In [ ]:
# Your custom request here
instructions = "You are a helpful assistant that can scrape and search the web."
request = "YOUR_REQUEST_HERE"  # Change this!
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="assistant", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("firecrawl-custom"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))